# SAT Hardness Engine — MVP Run
**Repositorio:** [pvsnp-investigacion-experimental](https://github.com/mcarrionsiete/pvsnp-investigacion-experimental)

Este notebook ejecuta el pipeline completo del SAT Hardness Engine:
1. Clona el repositorio
2. Instala dependencias
3. Genera instancias 3-SAT
4. Extrae features estructurales
5. Calcula hardness score v0.1
6. Ejecuta benchmark con Glucose4
7. Exporta CSV
8. Muestra correlaciones y primeras visualizaciones

**Ejecuta todas las celdas en orden** (Runtime > Run all)

## Paso 1 — Clonar repo e instalar dependencias

In [ ]:
!git clone https://github.com/mcarrionsiete/pvsnp-investigacion-experimental.git
%cd pvsnp-investigacion-experimental
!pip install python-sat[pblib,aiger] pandas numpy matplotlib scipy --quiet

## Paso 2 — Importar módulos del pipeline

In [ ]:
import sys
sys.path.insert(0, 'src')

from generator import generate_random_3sat
from features import extract_features
from scoring import compute_hardness_score
from benchmark import run_solver
from export import save_results
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

print('Modulos cargados correctamente')

## Paso 3 — Generar dataset de 1.000+ instancias

In [ ]:
configs = []
for n_vars in [20, 50, 100, 150, 200]:
 for alpha in [round(a * 0.1, 1) for a in range(20, 71)]:
 for seed in range(4):
 configs.append((n_vars, alpha, seed))

print(f'Total instancias a generar: {len(configs)}')

rows = []
for i, (n_vars, alpha, seed) in enumerate(configs):
 instance = generate_random_3sat(n_vars=n_vars, alpha=alpha, seed=seed)
 features = extract_features(instance)
 score = compute_hardness_score(features)
 bench = run_solver(instance)
 row = {'instance_id': f'inst_{i:05d}', 'seed': seed, **features, **score, **bench}
 rows.append(row)
 if (i+1) % 100 == 0:
 print(f' {i+1}/{len(configs)} instancias procesadas...')

df = save_results(rows, 'results/runs/mvp_run_v1.csv')
print(f'\nDataset guardado: {len(df)} instancias')
df.head()

## Paso 4 — Correlaciones: score vs métricas reales del solver

In [ ]:
cols = ['hardness_score_v1', 'solve_time_ms', 'conflicts', 'decisions', 'propagations']
corr = df[cols].corr(method='spearman')
print('=== Correlacion Spearman ===')
print(corr.round(3))

# Baseline trivial: solo distancia al punto critico
df['baseline_score'] = (1.0 - df['alpha_distance'] / 4.0).clip(0, 1)
print('\n=== Score v0.1 vs Baseline (correlacion con solve_time_ms) ===')
print(f"Score v0.1 : {df['hardness_score_v1'].corr(df['solve_time_ms'], method='spearman'):.4f}")
print(f"Baseline   : {df['baseline_score'].corr(df['solve_time_ms'], method='spearman'):.4f}")

## Paso 5 — Visualizaciones

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('SAT Hardness Engine — MVP Run v0.1', fontsize=15, fontweight='bold')

# 1. Score vs solve_time_ms
ax = axes[0, 0]
sample = df.sample(min(500, len(df)), random_state=42)
ax.scatter(sample['hardness_score_v1'], sample['solve_time_ms'], alpha=0.4, s=10, color='steelblue')
ax.set_xlabel('Hardness Score v0.1')
ax.set_ylabel('Solve Time (ms)')
ax.set_title('Score vs Tiempo de resolucion')
ax.set_yscale('log')

# 2. Tasa SAT por alpha (n=50)
ax = axes[0, 1]
df50 = df[df['n_vars'] == 50].copy()
sat_rate = df50.groupby('alpha')['sat'].mean()
ax.plot(sat_rate.index, sat_rate.values, color='tomato', linewidth=2)
ax.axvline(x=4.267, color='gray', linestyle='--', label='alpha_c = 4.267')
ax.set_xlabel('Alpha (clausulas/variables)')
ax.set_ylabel('Tasa SAT')
ax.set_title('Transicion de fase (n=50)')
ax.legend()

# 3. Hardness score por zona alpha
ax = axes[1, 0]
df50_grouped = df50.groupby('alpha')['hardness_score_v1'].mean()
ax.plot(df50_grouped.index, df50_grouped.values, color='mediumseagreen', linewidth=2)
ax.axvline(x=4.267, color='gray', linestyle='--', label='alpha_c = 4.267')
ax.set_xlabel('Alpha')
ax.set_ylabel('Hardness Score v0.1 (media)')
ax.set_title('Score por zona alpha (n=50)')
ax.legend()

# 4. Falsos positivos: score alto, tiempo bajo
ax = axes[1, 1]
q80_score = df['hardness_score_v1'].quantile(0.80)
q20_time = df['solve_time_ms'].quantile(0.20)
fp = df[(df['hardness_score_v1'] >= q80_score) & (df['solve_time_ms'] <= q20_time)]
fn_rate = len(fp) / len(df) * 100
ax.bar(['Resto', 'Falsos positivos'], [len(df) - len(fp), len(fp)],
 color=['steelblue', 'tomato'])
ax.set_title(f'Falsos positivos del score ({fn_rate:.1f}% del total)')
ax.set_ylabel('Instancias')

plt.tight_layout()
plt.savefig('results/figures/mvp_run_v1.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figura guardada en results/figures/mvp_run_v1.png')

## Paso 6 — Resumen del MVP

In [ ]:
print('=== RESUMEN MVP v0.1 ===')
print(f'Total instancias : {len(df)}')
print(f'Variables n : {sorted(df.n_vars.unique())}')
print(f'Alpha range : {df.alpha.min():.1f} - {df.alpha.max():.1f}')
print(f'SAT resueltos : {df.sat.sum()} ({df.sat.mean()*100:.1f}%)')
print(f'UNSAT : {(1-df.sat).sum()} ({(1-df.sat.mean())*100:.1f}%)')
print(f'Tiempo medio solver : {df.solve_time_ms.mean():.4f} ms')
print(f'Score medio v0.1 : {df.hardness_score_v1.mean():.4f}')
print(f'Correlacion score-tiempo : {df.hardness_score_v1.corr(df.solve_time_ms, method="spearman"):.4f}')
print(f'Correlacion baseline-tiempo: {df.baseline_score.corr(df.solve_time_ms, method="spearman"):.4f}')
print()
print('Archivos generados:')
print(' results/runs/mvp_run_v1.csv')
print(' results/figures/mvp_run_v1.png')
print()
print('Siguiente paso: abrir issue #4 y documentar estos resultados.')